# C. elegans Gene–Gene — Relation-Wise KG Triple Construction

## Purpose

This notebook processes Gene–Gene interaction data for *C. elegans* from **two sources** and combines them into standardized relation-wise Knowledge Graph (KG) triples:
1. **STRING** — Protein–protein interactions (mapped to genes via LocusTag)

## Pipeline Overview

| Step | Section | Description |
|------|---------|-------------|
| 1 | NCBI Gene Info | Load NCBI annotations, build LocusTag → Description dictionary |
| 2 | STRING Source | Load raw STRING file, clean protein IDs, map descriptions |
| 3 | WID/FICE Source | Load pre-processed pickle, add metadata, map descriptions |
| 4 | Combine Sources | Merge both into unified DataFrame with required columns |
| 5 | Validate, Deduplicate & Export | Quality check, deduplicate, export |

## Final Output Schema

| Column | Description |
|--------|-------------|
| `head` | Gene LocusTag |
| `relation` | `Gene_Gene` |
| `tail` | Gene LocusTag |
| `head_type` | `Gene` |
| `relation_type` | (NaN) |
| `tail_type` | `Gene` |
| `kg_source` | `STRING`, `FICE`, or `Worm_Interactome_Database` (merged with `::` on dedup) |
| `head_id_is` | `NCBI_ID` |
| `tail_id_is` | `NCBI_ID` |
| `head_detail_name` | Gene description from NCBI |
| `tail_detail_name` | Gene description from NCBI |
| `species` | `C.elegans` |

## Data Download Instructions

All databases used in this notebook must be downloaded prior to execution.
Please refer to the **central download instructions document** for detailed steps:

📄 **[Download Instructions — Link to be added]**

### Required Files

| File | Source | Description |
|------|--------|-------------|
| `6239.protein.links.detailed.v12.0.txt` | STRING | Raw protein–protein interaction file for C. elegans (tax_id: 6239) |
| `C_ELE_GENE_GENE_WithSEQ.pkl` | WID + FICE (pre-processed) | Gene–gene functional associations with sequences |
| `Caenorhabditis_elegans.gene_info` | NCBI Gene | Gene annotations for C. elegans |

---
## Setup

Import required libraries and define all base paths.

In [12]:
import pandas as pd
import numpy as np

In [13]:
# =============================================================================
# BASE PATHS — Update these to match your local directory structure
# =============================================================================
your_path_here = '/storage/Arushi/090526_EvoAge/kg_formation/data_collection/'

# NCBI gene info for C. elegans
NCBI_GENE_INFO_PATH = f'{your_path_here}databases_for_mapping/ncbi/Caenorhabditis_elegans.gene_info'

# Raw STRING protein-protein interaction file for C. elegans
STRING_RAW_PATH = f'{your_path_here}string/celegans/6239.protein.links.detailed.v12.0.txt'

# Pre-processed WID/FICE gene-gene pickle
# WID_FICE_PICKLE_PATH = '{your_path_here}/string/GENE_GENE/C_ELE_GENE_GENE_WithSEQ.pkl'

# Final output path
OUTPUT_PATH = '/storage/Arushi/090526_EvoAge/kg_formation/processed_data/string/celegans/string_CELE_GENE_GENE.csv'

---
## 1. NCBI Gene Info — Build LocusTag → Description Dictionary

Load NCBI gene annotations for *C. elegans*. Strip the `CELE_` prefix from LocusTags. In C. elegans, both STRING protein IDs and WID/FICE gene IDs correspond to LocusTags after cleaning.

In [14]:
# =============================================================================
# Load NCBI gene info and build dictionaries
# =============================================================================

ncbi_cele_gene = pd.read_csv(NCBI_GENE_INFO_PATH, sep='\t')

# Strip 'CELE_' prefix from LocusTag
ncbi_cele_gene['LocusTag'] = ncbi_cele_gene['LocusTag'].str.replace('CELE_', '', regex=False)

# Dictionary: LocusTag → Description (used for head/tail_detail_name)
ncbi_locustag_to_description = dict(zip(ncbi_cele_gene['LocusTag'], ncbi_cele_gene['description']))

print(f"NCBI genes loaded: {len(ncbi_cele_gene):,}")
print(f"LocusTag → Description dictionary size: {len(ncbi_locustag_to_description):,}")

NCBI genes loaded: 46,927
LocusTag → Description dictionary size: 46,927


In [15]:
ncbi_locustag_to_description

{'Y74C9A.3': 'Alpha N-terminal protein methyltransferase 1',
 'Y74C9A.2': 'Peptide P4',
 'Y74C9A.4': 'REST corepressor rcor-1',
 'Y74C9A.5': 'Sestrin homolog',
 'Y48G1C.4': 'CDP-diacylglycerol--glycerol-3-phosphate 3-phosphatidyltransferase',
 'Y48G1C.5': 'Tyrosine-protein phosphatase domain-containing protein',
 'Y48G1C.6': 'HTH CENPB-type domain-containing protein',
 'Y48G1C.1': 'Protein pid-2',
 'F53G12.1': 'Ras-related protein rab-11.1',
 'F53G12.10': 'Large ribosomal subunit protein uL30',
 'F53G12.9': 'GST_C_6 domain-containing protein;Metaxin glutathione S-transferase domain-containing protein',
 'F53G12.8': 'Uncharacterized protein',
 'F53G12.7': 'Collagen triple helix repeat protein;Nematode cuticle collagen N-terminal domain-containing protein',
 'F53G12.6': 'Spermatocyte protein spe-8',
 'F53G12.5': 'Growth-regulating factor;K Homology domain-containing protein',
 'F56C11.1': 'Dual oxidase 1',
 'F56C11.2': 'SSD domain-containing protein',
 'F56C11.6': 'Carboxylic ester hydro

---
## 2. STRING Source — Protein–Protein Interactions

Load the raw STRING file, strip species prefix, clean protein IDs (handle version numbers with exactly 2 dots), uppercase, and add KG metadata.

**ID cleaning logic:** C. elegans protein IDs can have version suffixes like `Y110A7A.10.1`. When there are exactly 2 dots, the middle segment is removed (e.g., `Y110A7A.10.1` → `Y110A7A.1`). IDs are then uppercased to match NCBI LocusTags.

In [16]:
# =============================================================================
# Load raw STRING protein-protein interaction file
# =============================================================================

string_ppi_df = pd.read_csv(STRING_RAW_PATH, sep='\\s')
string_ppi_df = string_ppi_df.rename(columns={'protein1': 'head', 'protein2': 'tail'})

# Strip species prefix '6239.' from both columns
string_ppi_df['tail'] = string_ppi_df['tail'].str.replace('6239.', '', regex=False)
string_ppi_df['head'] = string_ppi_df['head'].str.replace('6239.', '', regex=False)

print(f"Raw STRING interactions loaded: {len(string_ppi_df):,}")
string_ppi_df

# =============================================================================
# Clean protein IDs: handle version numbers and uppercase
# =============================================================================
# For IDs with exactly 2 dots (e.g., Y110A7A.10.1), remove the middle segment
# Then uppercase all IDs to match NCBI LocusTag format

for col in ['head', 'tail']:
    mask = string_ppi_df[col].str.count(r'\.') == 2
    string_ppi_df.loc[mask, col] = string_ppi_df.loc[mask, col].str.replace(r'\.[^.]+\.', '.', regex=True)

# Uppercase to match NCBI LocusTag format
string_ppi_df['head'] = string_ppi_df['head'].str.upper()
string_ppi_df['tail'] = string_ppi_df['tail'].str.upper()

print(f"Protein IDs cleaned and uppercased")
string_ppi_df

/tmp/ipykernel_2171772/803880184.py:5: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  string_ppi_df = pd.read_csv(STRING_RAW_PATH, sep='\\s')


Raw STRING interactions loaded: 9,271,876
Protein IDs cleaned and uppercased


,head,tail,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score
0,2L52.1,B0205.1,0,0,0,0,148,0,56,161
1,2L52.1,C24D10.1,0,0,0,0,220,0,0,220
2,2L52.1,C43E11.1,0,0,0,89,129,0,55,185
3,2L52.1,R13F6.1,0,0,0,0,167,0,0,167
4,2L52.1,C13D9.1,0,0,0,81,129,0,0,164
...,...,...,...,...,...,...,...,...,...,...
9271871,CTEL55X.1,F32A5.1,0,0,0,173,0,0,0,173
9271872,CTEL55X.1,F36A4.1,0,0,0,174,0,0,0,173
9271873,CTEL55X.1,Y73C8B.1,0,0,0,154,0,0,0,153
9271874,CTEL55X.1,C37C3.1,0,0,0,164,0,0,0,163


In [17]:
# =============================================================================
# Add KG metadata and map descriptions for STRING source
# =============================================================================
string_ppi_df['relation'] = 'Gene_Gene'
string_ppi_df['head_type'] = 'Gene'
string_ppi_df['tail_type'] = 'Gene'
string_ppi_df['relation_type'] = np.nan
string_ppi_df['kg_source'] = 'STRING'
string_ppi_df
string_ppi_df['head_detail_name'] = string_ppi_df['head'].map(ncbi_locustag_to_description)
string_ppi_df['tail_detail_name'] = string_ppi_df['tail'].map(ncbi_locustag_to_description)

In [18]:
string_ppi_df

,head,tail,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,relation,head_type,tail_type,relation_type,kg_source,head_detail_name,tail_detail_name
0,2L52.1,B0205.1,0,0,0,0,148,0,56,161,Gene_Gene,Gene,Gene,NaN,STRING,C2H2-type domain-containing protein,SPK domain-containing protein
1,2L52.1,C24D10.1,0,0,0,0,220,0,0,220,Gene_Gene,Gene,Gene,NaN,STRING,C2H2-type domain-containing protein,Tyrosine-protein phosphatase domain-containing...
2,2L52.1,C43E11.1,0,0,0,89,129,0,55,185,Gene_Gene,Gene,Gene,NaN,STRING,C2H2-type domain-containing protein,SAP domain-containing protein
3,2L52.1,R13F6.1,0,0,0,0,167,0,0,167,Gene_Gene,Gene,Gene,NaN,STRING,C2H2-type domain-containing protein,KNL (kinetochore null) Binding Protein
4,2L52.1,C13D9.1,0,0,0,81,129,0,0,164,Gene_Gene,Gene,Gene,NaN,STRING,C2H2-type domain-containing protein,ABC transmembrane type-1 domain-containing pro...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9271871,CTEL55X.1,F32A5.1,0,0,0,173,0,0,0,173,Gene_Gene,Gene,Gene,NaN,STRING,Androgen-induced gene 1 protein;FAR-17a/AIG1-l...,SANT domain-containing protein
9271872,CTEL55X.1,F36A4.1,0,0,0,174,0,0,0,173,Gene_Gene,Gene,Gene,NaN,STRING,Androgen-induced gene 1 protein;FAR-17a/AIG1-l...,Uncharacterized protein
9271873,CTEL55X.1,Y73C8B.1,0,0,0,154,0,0,0,153,Gene_Gene,Gene,Gene,NaN,STRING,Androgen-induced gene 1 protein;FAR-17a/AIG1-l...,AB hydrolase-1 domain-containing protein
9271874,CTEL55X.1,C37C3.1,0,0,0,164,0,0,0,163,Gene_Gene,Gene,Gene,NaN,STRING,Androgen-induced gene 1 protein;FAR-17a/AIG1-l...,DUF1768 domain-containing protein;NADAR domain...


In [19]:

string_ppi_df['head_id_is'] = 'NCBI_ID'
string_ppi_df['tail_id_is'] = 'NCBI_ID'
string_ppi_df['species'] = 'C.elegans'
string_ppi_df


,head,tail,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,relation,head_type,tail_type,relation_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,species
0,2L52.1,B0205.1,0,0,0,0,148,0,56,161,Gene_Gene,Gene,Gene,NaN,STRING,C2H2-type domain-containing protein,SPK domain-containing protein,NCBI_ID,NCBI_ID,C.elegans
1,2L52.1,C24D10.1,0,0,0,0,220,0,0,220,Gene_Gene,Gene,Gene,NaN,STRING,C2H2-type domain-containing protein,Tyrosine-protein phosphatase domain-containing...,NCBI_ID,NCBI_ID,C.elegans
2,2L52.1,C43E11.1,0,0,0,89,129,0,55,185,Gene_Gene,Gene,Gene,NaN,STRING,C2H2-type domain-containing protein,SAP domain-containing protein,NCBI_ID,NCBI_ID,C.elegans
3,2L52.1,R13F6.1,0,0,0,0,167,0,0,167,Gene_Gene,Gene,Gene,NaN,STRING,C2H2-type domain-containing protein,KNL (kinetochore null) Binding Protein,NCBI_ID,NCBI_ID,C.elegans
4,2L52.1,C13D9.1,0,0,0,81,129,0,0,164,Gene_Gene,Gene,Gene,NaN,STRING,C2H2-type domain-containing protein,ABC transmembrane type-1 domain-containing pro...,NCBI_ID,NCBI_ID,C.elegans
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9271871,CTEL55X.1,F32A5.1,0,0,0,173,0,0,0,173,Gene_Gene,Gene,Gene,NaN,STRING,Androgen-induced gene 1 protein;FAR-17a/AIG1-l...,SANT domain-containing protein,NCBI_ID,NCBI_ID,C.elegans
9271872,CTEL55X.1,F36A4.1,0,0,0,174,0,0,0,173,Gene_Gene,Gene,Gene,NaN,STRING,Androgen-induced gene 1 protein;FAR-17a/AIG1-l...,Uncharacterized protein,NCBI_ID,NCBI_ID,C.elegans
9271873,CTEL55X.1,Y73C8B.1,0,0,0,154,0,0,0,153,Gene_Gene,Gene,Gene,NaN,STRING,Androgen-induced gene 1 protein;FAR-17a/AIG1-l...,AB hydrolase-1 domain-containing protein,NCBI_ID,NCBI_ID,C.elegans
9271874,CTEL55X.1,C37C3.1,0,0,0,164,0,0,0,163,Gene_Gene,Gene,Gene,NaN,STRING,Androgen-induced gene 1 protein;FAR-17a/AIG1-l...,DUF1768 domain-containing protein;NADAR domain...,NCBI_ID,NCBI_ID,C.elegans


In [20]:

# Filter out unmapped entries
string_ppi_df = string_ppi_df[~string_ppi_df['head_detail_name'].isna()]
string_ppi_df = string_ppi_df[~string_ppi_df['tail_detail_name'].isna()]

print(f"STRING Gene-Gene triples after filtering: {len(string_ppi_df):,}")
string_ppi_df
string_ppi_df

STRING Gene-Gene triples after filtering: 7,867,356


,head,tail,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,relation,head_type,tail_type,relation_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,species
0,2L52.1,B0205.1,0,0,0,0,148,0,56,161,Gene_Gene,Gene,Gene,NaN,STRING,C2H2-type domain-containing protein,SPK domain-containing protein,NCBI_ID,NCBI_ID,C.elegans
1,2L52.1,C24D10.1,0,0,0,0,220,0,0,220,Gene_Gene,Gene,Gene,NaN,STRING,C2H2-type domain-containing protein,Tyrosine-protein phosphatase domain-containing...,NCBI_ID,NCBI_ID,C.elegans
2,2L52.1,C43E11.1,0,0,0,89,129,0,55,185,Gene_Gene,Gene,Gene,NaN,STRING,C2H2-type domain-containing protein,SAP domain-containing protein,NCBI_ID,NCBI_ID,C.elegans
3,2L52.1,R13F6.1,0,0,0,0,167,0,0,167,Gene_Gene,Gene,Gene,NaN,STRING,C2H2-type domain-containing protein,KNL (kinetochore null) Binding Protein,NCBI_ID,NCBI_ID,C.elegans
4,2L52.1,C13D9.1,0,0,0,81,129,0,0,164,Gene_Gene,Gene,Gene,NaN,STRING,C2H2-type domain-containing protein,ABC transmembrane type-1 domain-containing pro...,NCBI_ID,NCBI_ID,C.elegans
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9271871,CTEL55X.1,F32A5.1,0,0,0,173,0,0,0,173,Gene_Gene,Gene,Gene,NaN,STRING,Androgen-induced gene 1 protein;FAR-17a/AIG1-l...,SANT domain-containing protein,NCBI_ID,NCBI_ID,C.elegans
9271872,CTEL55X.1,F36A4.1,0,0,0,174,0,0,0,173,Gene_Gene,Gene,Gene,NaN,STRING,Androgen-induced gene 1 protein;FAR-17a/AIG1-l...,Uncharacterized protein,NCBI_ID,NCBI_ID,C.elegans
9271873,CTEL55X.1,Y73C8B.1,0,0,0,154,0,0,0,153,Gene_Gene,Gene,Gene,NaN,STRING,Androgen-induced gene 1 protein;FAR-17a/AIG1-l...,AB hydrolase-1 domain-containing protein,NCBI_ID,NCBI_ID,C.elegans
9271874,CTEL55X.1,C37C3.1,0,0,0,164,0,0,0,163,Gene_Gene,Gene,Gene,NaN,STRING,Androgen-induced gene 1 protein;FAR-17a/AIG1-l...,DUF1768 domain-containing protein;NADAR domain...,NCBI_ID,NCBI_ID,C.elegans


In [21]:

# Keep + reorder columns
string_ppi_df = string_ppi_df[
    [
        "head",
        "relation",
        "tail",
        "head_type",
        "relation_type",
        "tail_type",
        "kg_source",
        "head_id_is",
        "tail_id_is",
        "head_detail_name",
        "tail_detail_name",
        "species"
    ]
]

# Add kg_type column
string_ppi_df["kg_type"] = string_ppi_df["species"]

# Final column order
string_ppi_df = string_ppi_df[
    [
        "head",
        "relation",
        "tail",
        "head_type",
        "relation_type",
        "tail_type",
        "kg_source",
        "kg_type",
        "head_id_is",
        "tail_id_is",
        "head_detail_name",
        "tail_detail_name",
        "species"
    ]
]

# Save
string_ppi_df

/tmp/ipykernel_2171772/3154585268.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  string_ppi_df["kg_type"] = string_ppi_df["species"]


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,2L52.1,Gene_Gene,B0205.1,Gene,NaN,Gene,STRING,C.elegans,NCBI_ID,NCBI_ID,C2H2-type domain-containing protein,SPK domain-containing protein,C.elegans
1,2L52.1,Gene_Gene,C24D10.1,Gene,NaN,Gene,STRING,C.elegans,NCBI_ID,NCBI_ID,C2H2-type domain-containing protein,Tyrosine-protein phosphatase domain-containing...,C.elegans
2,2L52.1,Gene_Gene,C43E11.1,Gene,NaN,Gene,STRING,C.elegans,NCBI_ID,NCBI_ID,C2H2-type domain-containing protein,SAP domain-containing protein,C.elegans
3,2L52.1,Gene_Gene,R13F6.1,Gene,NaN,Gene,STRING,C.elegans,NCBI_ID,NCBI_ID,C2H2-type domain-containing protein,KNL (kinetochore null) Binding Protein,C.elegans
4,2L52.1,Gene_Gene,C13D9.1,Gene,NaN,Gene,STRING,C.elegans,NCBI_ID,NCBI_ID,C2H2-type domain-containing protein,ABC transmembrane type-1 domain-containing pro...,C.elegans
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9271871,CTEL55X.1,Gene_Gene,F32A5.1,Gene,NaN,Gene,STRING,C.elegans,NCBI_ID,NCBI_ID,Androgen-induced gene 1 protein;FAR-17a/AIG1-l...,SANT domain-containing protein,C.elegans
9271872,CTEL55X.1,Gene_Gene,F36A4.1,Gene,NaN,Gene,STRING,C.elegans,NCBI_ID,NCBI_ID,Androgen-induced gene 1 protein;FAR-17a/AIG1-l...,Uncharacterized protein,C.elegans
9271873,CTEL55X.1,Gene_Gene,Y73C8B.1,Gene,NaN,Gene,STRING,C.elegans,NCBI_ID,NCBI_ID,Androgen-induced gene 1 protein;FAR-17a/AIG1-l...,AB hydrolase-1 domain-containing protein,C.elegans
9271874,CTEL55X.1,Gene_Gene,C37C3.1,Gene,NaN,Gene,STRING,C.elegans,NCBI_ID,NCBI_ID,Androgen-induced gene 1 protein;FAR-17a/AIG1-l...,DUF1768 domain-containing protein;NADAR domain...,C.elegans


In [ ]:
# =============================================================================
# Export final deduplicated Chemical-Gene KG triples
# =============================================================================

string_ppi_df.to_csv(OUTPUT_PATH, index=None)
print(f"Final output saved to: {OUTPUT_PATH}")

In [ ]:
OUTPUT_PATH